# Módulo 01: Inicialización Canónica de SparkSession y Arquitectura Moderna

## 1. Evolución de la API: Paradigma Clásico vs. Estándar Moderno

En las versiones clásicas de Apache Spark (Spark 1.x y 2.x), los desarrolladores debían lidiar con múltiples puntos de entrada desacoplados y fragmentados:
* **`SparkContext` (`sc`)**: Exclusivo para operaciones de bajo nivel con RDDs.
* **`SQLContext`**: Diseñado para interactuar con tablas estructuradas tempranas.
* **`HiveContext`**: Requerido si se necesitaba acceso a catálogos persistentes de Apache Hive.
* **`StreamingContext`**: Instancia aislada para micro-batch streaming tradicional.

### El Estándar Unificado: `SparkSession`
A partir de Spark 3.x, el punto de entrada oficial y canónico es **`SparkSession`**, implementado mediante el patrón de diseño *Builder*. Esta abstracción unifica el acceso a:
1. **DataFrames y Datasets**: Procesamiento estructurado optimizado por Catalyst.
2. **Spark SQL**: Ejecución de sentencias ANSI SQL sobre catálogos y almacenes de datos.
3. **Structured Streaming**: Ingesta continua basada en tablas y marcas de agua (*watermarks*).
4. **Spark Connect**: Arquitectura cliente-servidor desacoplada mediante búferes de protocolo (gRPC), separando el driver local del clúster de cómputo.S

In [1]:
from pyspark.sql import SparkSession

# Inicialización canónica recomendada por la documentación oficial
spark = (
    SparkSession.builder
    .appName("01_Sesion_Canonica")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .getOrCreate()
)

print(f"Versión activa de Apache Spark: {spark.version}")
spark

Versión activa de Apache Spark: 3.5.9


## 2. Inspección Interna del Motor en Tiempo de Ejecución

Una de las ventajas clave de `SparkSession` frente a los enfoques legados es la capacidad de consultar, inspeccionar y modificar configuraciones dinámicas de la sesión en caliente mediante el gestor `spark.conf`, sin necesidad de reiniciar la JVM subyacente.

En entornos de pruebas locales o recursos limitados, ajustar parámetros como `spark.sql.shuffle.partitions` evita que Spark instancie por defecto 200 particiones vacías durante operaciones de agregación o joins.

In [2]:
# Consulta programática de variables dinámicas activas
master_actual = spark.conf.get("spark.master")
shuffle_actual = spark.conf.get("spark.sql.shuffle.partitions")

print(f"Modo de asignación (Master): {master_actual}")
print(f"Particiones asignadas para etapas de Shuffle: {shuffle_actual}")

# Verificación de parámetros internos del SparkContext subyacente
sc = spark.sparkContext
print(f"Identificador de la aplicación en el gestor: {sc.applicationId}")
print(f"Versión de Python utilizada por el Driver: {sc.pythonVer}")

Modo de asignación (Master): local[*]
Particiones asignadas para etapas de Shuffle: 4
Identificador de la aplicación en el gestor: local-1788728388086
Versión de Python utilizada por el Driver: 3.11


## 3. Spark Connect: La Arquitectura Desacoplada

Tradicionalmente, el proceso *Driver* de PySpark debía correr en la misma red física o máquina con la JVM local. Si la conexión de red sufría latencia o caídas de socket, el trabajo entero fallaba.

### ¿Cómo opera Spark Connect?
* **Lado Cliente (Lightweight)**: PySpark opera como un cliente liviano sin requerir binarios pesados ni dependencias nativas complejas en la máquina cliente.
* **Protocolo gRPC**: Las transformaciones se serializan en planes lógicos no resueltos y se transmiten vía gRPC hacia un servidor de Spark Connect.
* **Lado Servidor**: El servidor Spark Connect traduce la consulta, la optimiza con Catalyst y distribuye el cómputo en el clúster.

En un entorno local o contenedor educativo, la instancia canónica de `SparkSession.builder` provee el entorno idéntico para que el código escrito sea 100% compatible tanto en local como al conectarse a un clúster remoto vía `sc.connect("sc://servidor:15002")`.

In [3]:
# Creación de datos distribuidos en memoria mediante APIs nativas
df_base = spark.range(1, 1000, step=2)

# Inspección de metadatos del DataFrame sin disparar cómputo innecesario (Lazy Evaluation)
print("Esquema inferido formalmente:")
df_base.printSchema()

print(f"Número de particiones físicas iniciales: {df_base.rdd.getNumPartitions()}")

Esquema inferido formalmente:
root
 |-- id: long (nullable = false)

Número de particiones físicas iniciales: 4


In [6]:
df_base.show(3)

+---+
| id|
+---+
|  1|
|  3|
|  5|
+---+
only showing top 3 rows



## 4. Reto Práctico: Agregaciones Nativas y Validación con Aserciones

### Instrucciones del Ejercicio:
1. Utiliza `spark.range()` para generar una secuencia numérica que empiece en `1` y termine en `100` (con paso de `1`).
2. Filtra la secuencia para conservar **únicamente los números que sean múltiplos de 3**.
3. Calcula la **suma total** de dichos números utilizando la función `F.sum()` de `pyspark.sql.functions`.
4. Extrae el resultado escalar y asegúrate de que supere la aserción automática.

In [7]:
import pyspark.sql.functions as F

# 1. Creación del rango del 1 al 100 inclusive
df_rango = spark.range(1, 101)

# 2. Filtrado de múltiplos de 3 y cálculo de agregación
resultado_df = (
    df_rango
    .filter(F.col("id") % 3 == 0)
    .select(F.sum("id").alias("suma_multiplos_3"))
)

# 3. Extracción del escalar
total_calculado = resultado_df.collect()[0]["suma_multiplos_3"]
print(f"Resultado obtenido: {total_calculado}")

# 4. Validación automática (La suma de múltiplos de 3 entre 1 y 100 es 1683)
assert total_calculado == 1683, f"Aserción fallida: Se esperaba 1683 pero se obtuvo {total_calculado}"
print("¡Aserción aprobada! Cuaderno 01 completado exitosamente.")

Resultado obtenido: 1683
¡Aserción aprobada! Cuaderno 01 completado exitosamente.
